# 🧪 Sesión 25 · Consultas Complejas, Joins y Carga Ansiosa (Eager Loading)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué una lista de 100 autores dispara 101 consultas?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/autores
HTTP/1.1 200 OK
{"autores":[
  {"id":1,"nombre":"Gabriel García Márquez",
   "libros":[{"id":1,"titulo":"Cien años de soledad"}]},
  {"id":2,"nombre":"Julio Cortázar",
   "libros":[{"id":2,"titulo":"Rayuela"}]}
]}

GET /api/autores/1
HTTP/1.1 200 OK
{"id":1,"nombre":"Gabriel García Márquez",
 "libros":[{"id":1,"titulo":"Cien años de soledad"}]}
```

_El resultado es el mismo; la diferencia es cuántas consultas hiciste para lograrlo._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Cargar relaciones con selectinload.
- Evitar el problema N+1.
- Verificar que la respuesta anida los libros del autor.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Los autores llegan con sus libros → HTTP 200
- [ ] Un autor concreto trae sus libros → HTTP 200
- [ ] El listado de libros sigue disponible → HTTP 200
- [ ] Se usa carga ansiosa

## 🧩 El reto

Implementa una consulta de autores con sus libros usando carga ansiosa para evitar el problema N+1.

**Requisitos del contrato**

- Siembra exactamente: Gabriel García Márquez → «Cien años de soledad»; Julio Cortázar → «Rayuela».
- GET /api/autores devuelve todos los autores con sus libros anidados.
- GET /api/autores/{id} devuelve un autor con sus libros.
- La consulta usa selectinload o joinedload.
- La respuesta no incluye autores sin sus libros cuando existen.
- El listado de libros sigue disponible en /api/libros.

> **Pista conceptual:** Sin carga ansiosa, cada autor dispara una consulta para sus libros. `selectinload(Autor.libros)` resuelve la relación en una sola consulta adicional.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion25.md` explicando el problema N+1.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/main/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tu consulta ansiosa se verá así")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'Los autores llegan con sus libros',
  'metodo': 'GET',
  'ruta': '/api/autores',
  'estado': 200,
  'json_contiene': {'autores': [{'id': 1,
                                 'nombre': 'Gabriel García Márquez',
                                 'libros': [{'id': 1, 'titulo': 'Cien años de soledad'}]},
                                {'id': 2, 'nombre': 'Julio Cortázar', 'libros': [{'id': 2, 'titulo': 'Rayuela'}]}]}},
 {'descripcion': 'Un autor concreto trae sus libros',
  'metodo': 'GET',
  'ruta': '/api/autores/1',
  'estado': 200,
  'json_contiene': {'id': 1, 'libros': [{'id': 1, 'titulo': 'Cien años de soledad'}]}},
 {'descripcion': 'El listado de libros sigue disponible',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}, {'id': 2}]}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["selectinload|joinedload"], descripcion="Se usa carga ansiosa"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion25.md` a `auditorias/auditoria_sesion25.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
